In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
from keras.callbacks import EarlyStopping
import joblib

In [ ]:
# Stock symbol
stock = "GOOG"

# Time range: last 20 years
end = datetime.now()
start = datetime(end.year - 20, end.month, end.day)

# Download data
google_data = yf.download(stock, start=start, end=end, auto_adjust=False, progress=False)

# Check for valid data
if google_data.empty:
    raise ValueError("No data returned. Check ticker or network connection.")
if 'Adj Close' not in google_data.columns:
    raise ValueError("'Adj Close' column is missing. Try again later or check ticker.")


In [ ]:
# Display basic info
print(google_data.describe())
print(google_data.info())
print(google_data.isna().sum())

# Plotting function
def plot_graph(figsize, values, column_name):
    plt.figure(figsize=figsize)
    if isinstance(values, pd.DataFrame):
        for col in values.columns:
            plt.plot(values[col], label=col)
        plt.legend()
    else:
        plt.plot(values)
    plt.xlabel("Years")
    plt.ylabel(column_name)
    plt.title(f"{column_name} of Google data")
    plt.grid(True)
    plt.show()

# Plot all columns
for column in google_data.columns:
    plot_graph((15, 5), google_data[column], column)

In [ ]:
# Moving averages
google_data['MA_for_100_days'] = google_data['Adj Close'].rolling(100).mean()
google_data['MA_for_250_days'] = google_data['Adj Close'].rolling(250).mean()

# Plot moving averages
plot_graph((15, 5), google_data[['Adj Close', 'MA_for_100_days', 'MA_for_250_days']], 'Moving Averages')

In [ ]:
# Percentage change
google_data['percentage_change_cp'] = google_data['Adj Close'].pct_change()
plot_graph((15, 5), google_data['percentage_change_cp'], 'Percentage Change')


In [ ]:
# Prepare data
Adj_close_price = google_data[['Adj Close']].dropna()

# Normalize
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(Adj_close_price)

# Sequence creation
x_data, y_data = [], []
for i in range(100, len(scaled_data)):
    x_data.append(scaled_data[i-100:i])
    y_data.append(scaled_data[i])

x_data, y_data = np.array(x_data), np.array(y_data)
x_data = x_data.reshape((x_data.shape[0], x_data.shape[1], 1))

# Split train/test
split_len = int(len(x_data) * 0.7)
x_train, y_train = x_data[:split_len], y_data[:split_len]
x_test, y_test = x_data[split_len:], y_data[split_len:]

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)


In [ ]:
# Build model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    LSTM(64, return_sequences=False),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

# Train model
early_stop = EarlyStopping(monitor='loss', patience=3)
model.fit(x_train, y_train, batch_size=32, epochs=10, callbacks=[early_stop])
model.summary()

In [ ]:
# Predictions
predictions = model.predict(x_test)
inv_predictions = scaler.inverse_transform(predictions)
inv_y_test = scaler.inverse_transform(y_test)

# RMSE
rmse = np.sqrt(np.mean((inv_predictions - inv_y_test) ** 2))
print(f"RMSE: {rmse}")

In [ ]:
# Plot predictions
ploting_data = pd.DataFrame({
    'original_test_data': inv_y_test.reshape(-1),
    'predictions': inv_predictions.reshape(-1)
}, index=Adj_close_price.index[split_len + 100:])

plot_graph((15, 6), ploting_data, 'Test Data: Actual vs Predicted')

# Full view
combined = pd.concat([Adj_close_price[:split_len + 100], ploting_data], axis=0)
plot_graph((15, 6), combined, 'Full View: Actual vs Predicted')


In [ ]:
# Save model and scaler
model.save("Latest_stock_price_model.keras")
joblib.dump(scaler, "price_scaler.pkl")